# പാഠം 03 - ഏജന്റിക് ഡിസൈൻ പാറ്റേൺസ്

ഈ പാഠത്തിൽ, ഫലപ്രദമായ AI ഏജന്റുകൾ നിർമ്മിക്കാൻ മൂന്ന് അടിസ്ഥാന ഡിസൈൻ പാറ്റേൺസ് പരിശോധിക്കുന്നു:

1. **പഴയ്‌റ്റ ഏജന്റ് നിര്‍ദ്ദേശങ്ങൾ** — ഏജന്റ് പെരുമാറ്റം നയിക്കുന്ന വ്യക്തമായ, പങ്ക് നിർവചിക്കുന്ന പ്രോമ്പ്റ്റുകൾ രൂപകല്‍പ്പന ചെയ്യുക  
2. **പൈഡാന്റ്റിക് മോഡലുകളോടെ ഘടിത ഔട്ട്‌പുട്ട്** — ഏജന്റുകൾ പ്രവചിക്കാവുന്ന, സാധൂകരിച്ച ഡാറ്റ മടക്കുന്നത് ഉറപ്പാക്കല്‍  
3. **ഏക ദ്വിതീയ ഉത്തരവാദിത്വമുള്ള ഏജന്റുകൾ** — ഓരോന്നും ഒരു കാര്യം നന്നായി ചെയ്യുവാനുള്ള കേന്ദ്രീകൃത ഏജന്റുകൾ രൂപകല്‍പ്പന ചെയ്യൽ  

ഓരോ പാറ്റേണും **യാത്രാ സ്ഥല ശുപാർശാ സിസ്റ്റം** ആപ്ലിക്കേഷൻ സന്ധർഭത്തിൽ ആവിഷ്ക്കരിച്ച്, ക്രമാനുസൃതമായി സ്ഥലങ്ങൾ ശുപാർശ ചെയ്യാൻ, ലഭ്യത പരിശോധിക്കാൻ, ലജിസ്റ്റിക്സ് കൈകാര്യം ചെയ്യാൻ കഴിയുന്ന ഒരു സിസ്റ്റം നിർമ്മിച്ചുകൊള്ളും.


## സജ്ജമാക്കൽ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## മാതൃക 1: വ്യക്തമായ ഏജന്റ് നിർദ്ദേശങ്ങൾ

ഏറ്റവും പ്രഭാവശാലിയായ മാതൃക എന്നും എളുപ്പമാണ്: നിങ്ങളുടെ ഏജന്റിന് വ്യക്തമായ, വിശദമായ നിർദ്ദേശങ്ങൾ എഴുതുക.

നല്ല നിർദ്ദേശങ്ങൾ നിർവചിക്കുന്നത്:
- **ആരെ** ആണ് ഏജന്റ് (വ്യക്തിത്വവും ശൈലിയും)
- **എന്ത്** ചെയ്യണം (പടി ബൈ പടി ഉത്തരവാദിത്വങ്ങൾ)
- **എങ്ങനെ** പെരുമാറണം (നിയമങ്ങളും ശൈലിയുമടങ്ങി)

താഴെ, ഓരോ പ്രതികരണത്തിനും രൂപകൽപ്പന ചെയ്യുന്ന വ്യക്തമായ നിർദ്ദേശങ്ങളോടുകൂടെ ഒരു യാത്രാ കോൺസിയോർജ് ഏജന്റ് സൃഷ്ടിക്കുന്നു.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic മോഡലുകൾ ഉപയോഗിച്ച് ഘടനാപരമായ ഔട്ട്‌പുട്ട്

സ്വതന്ത്ര രൂപത്തിലുള്ള വാചകം സംഭാഷണത്തിന് ഉപകാരപ്രദമാണെങ്കിലും, താഴെ പ്രവർത്തിക്കുന്ന സിസ്റ്റങ്ങൾ ഘടനാപരമായ ഡാറ്റ ആവശ്യമാണ്.  
**Pydantic മോഡലുകൾ** ഒരു **ടൂൾ ഫംഗ്ഷനുമായി** കൂട്ടിച്ചേർത്താൽ, നാം:

- ഏജന്റിന്റെ ഔട്ട്‌പുട്ടിന് കൃത്യമായ സ്കീമ നിർവ്വചിക്കുക  
- പ്രതികരണങ്ങൾ സ്വയം സ്ഥിരീകരിക്കുക  
- ഏജന്റ് ഫലങ്ങൾ ആപ്ലിക്കേഷൻ ലജിക്കിലേക്ക് വിശ്വാസാർഹമായി സംയോജിപ്പിക്കുക  

നാം ഒരു ടൂൾ കൂടി പരിചയപ്പെടുത്തുന്നു, അത് ലക്ഷ്യസ്ഥല വിശദാംശങ്ങൾ തിരിച്ചറിഞ്ഞ് ഏജന്റ് നിർദ്ദേശങ്ങൾ യഥാർത്ഥ ഡാറ്റയിൽ അടിസ്ഥാനം പിടിക്കുന്നതു ഉറപ്പാക്കുന്നു.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Single Responsibility Agents

സങ്കീർണ പ്രവർത്തനങ്ങൾ പല ശ്രദ്ധിച്ചിരിക്കുന്ന ഏജന്റുമാർമാർ തമ്മിൽ ജോലികൾ വിഭജിച്ച് ഫലം ചെയ്യും, ഓരോരുത്തരുടെയും ഒരു മാത്രമാണ് ഉത്തരവാദിത്വം:

- ഇടങ്ങൾക്കും ലഭ്യതയ്ക്കുമുള്ള അറിവുള്ള **ആഭിപ്രായ വിദഗ്ധൻ**
- പറക്കൽ, ഹോട്ടലുകൾ, യാത്രാവിവരങ്ങൾ കൈകാര്യം ചെയ്യുന്ന **ലൊജിസ്റ്റിക്സ് പ്ലാനർ**

ഇത് സോഫ്റ്റ്‌വെയർ എൻജിനീയറിംഗ് സിദ്ധാന്തമായ *വിഭജനം* എന്ന ആശയത്തിന് ഐക്യമായി പ്രവർത്തിക്കുന്നു — ഓരോ ഏജന്റും സ്വതന്ത്രമായി പരിശോധിക്കാൻ, പരിപാലിക്കാൻ, മെച്ചപ്പെടുത്താൻ എളുപ്പമാണ്.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## സംക്ഷેપം

ഈ പാഠത്തിലാണ് നാം യാത്രാ ശിപാര്‍ശ വ്യവസ്ഥയില്‍ മൂന്ന് ഏജന്റിക് ഡിസൈന്‍ പാറ്റേണുകള്‍ പ്രയോഗിച്ചത്:

| പാറ്റേണ്‍ | പ്രധാന ആശയം | പ്രയോജനം |
|---|---|---|
| **തിരോധമായ നിര്‍ദേശങ്ങള്‍** | വ്യക്തിത്വം, ഉത്തരവാദിത്വങ്ങള്‍, നിയന്ത്രണങ്ങള്‍ മുൻകൂറായി നിർവചിക്കുക | സ്ഥിരതയുള്ള, ബ്രാന്‍ഡായി ഏജന്റിന്റെ പെരുമാറ്റം |
| **സംഘടിത ഔട്ട്പുട്** | പ്രതികരണ ഫോര്‍മാറ്റായി പിഡാന്റിക് മോഡലുകള്‍ ഉപയോഗിക്കുക | പരിശോധനയുള്ള, യന്ത്രം വായിക്കാൻ കഴിയുന്ന ഫലങ്ങൾ |
| **ഏക ഉത്തരവാദിത്വം** | ഓരോ ഏജന്റിനും ഒരു കേന്ദ്രീകൃത ജോലി നൽകുക | പരീക്ഷിക്കുക, പരിപാലിക്കുക, സംയೋಜിക്കുക എളുപ്പം |

ഈ പാറ്റേണുകള്‍ സ്വാഭാവികമായി സംയോജിക്കുന്നു — ഒരു ഏക ഉത്തരവാദിത്വ ഏജന്റിൽ വ്യക്തമാക്കിയ നിർദേശങ്ങൾ, സംഘടിത ഔട്ട്പുട്ട് എന്നിവ സംയോജിപ്പിച്ച് ശക്തമായ, ഉല്‍പ്പാദന מוכൻ സിസ്റ്റങ്ങൾ നിർമ്മിക്കാം.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**റദ്ദാക്കൽ**:  
ഈ പ്രമാണം AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. കൃത്യതയ്ക്ക് ഞങ്ങൾ പരിശ്രമിച്ചിട്ടുണ്ടെങ്കിലും, ഓട്ടോമേറ്റഡ് വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ അശുദ്ധികൾ ഉണ്ടായിരിക്കാമെന്ന് ദയവായി അറിയുക. ഇതിന്റെ മാതൃഭാഷയിലെ 원본 പ്രമാണം അഥോറിറ്റേറ്റിവ് സ്രോതസ്സായി കണക്കാക്കേണ്ടതാണ്. നിർണായക വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ഉപദേശം നൽകുന്നു. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിനാൽ ഉണ്ടാകുന്ന യാതൊരു തെറ്റിദ്ധാരണക്കും ഞങ്ങൾ ഉത്തരവാദിത്വം ഏറ്റെടുക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
